# Week 1 — Introduction to Data Engineering
## Hands-on project: implement a simple ETL pipeline in Python

This mirrors the Week 1 assignment: fork a repo, implement ETL, submit for
auto-grading. Here the "auto-grading" is a set of `assert` checks at the
bottom of the notebook — run all cells; if the last cell prints
`ALL CHECKS PASSED`, the pipeline is correct.

### Step 1: Extract
Read the raw order data.

In [1]:
import pandas as pd

RAW_PATH = "../../02_etl_pipeline/sample_data/orders_raw.csv"

def extract_orders(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)
    print(f"[extract] read {len(df)} raw rows")
    return df

raw = extract_orders(RAW_PATH)
raw.head()

[extract] read 10 raw rows


,order_id,customer_name,email,product,quantity,unit_price,order_date,country
0,1001,Alice Ng,alice@example.com,Widget A,3,9.99,2026-01-05,South Africa
1,1002,Bob Smith,bob@example.com,Widget B,1,24.50,2026-01-06,USA
2,1003,NaN,carol@example.com,Widget A,2,9.99,2026-01-06,UK
3,1004,Dana Lee,dana@example.com,Widget C,5,4.25,2026-01-07,South Africa
4,1005,Evan Cho,evan@example.com,Widget B,NaN,24.50,2026-01-07,Kenya


### Step 2: Transform
Clean types, quarantine bad rows, derive `total_price`.

In [2]:
def transform_orders(raw: pd.DataFrame):
    df = raw.copy()
    for col in ["customer_name", "email", "product", "country"]:
        df[col] = df[col].astype(str).str.strip().replace({"nan": None, "": None})

    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    required = ["customer_name", "email", "product", "quantity", "unit_price", "order_date"]
    is_valid = df[required].notna().all(axis=1)

    clean = df[is_valid].copy()
    rejected = df[~is_valid].copy()
    clean["total_price"] = (clean["quantity"] * clean["unit_price"]).round(2)
    clean["order_date"] = clean["order_date"].dt.date.astype(str)

    print(f"[transform] {len(clean)} valid rows, {len(rejected)} rejected rows")
    return clean.reset_index(drop=True), rejected.reset_index(drop=True)

clean, rejected = transform_orders(raw)
clean

[transform] 7 valid rows, 3 rejected rows


,order_id,customer_name,email,product,quantity,unit_price,order_date,country,total_price
0,1001,Alice Ng,alice@example.com,Widget A,3.0,9.99,2026-01-05,South Africa,29.97
1,1002,Bob Smith,bob@example.com,Widget B,1.0,24.50,2026-01-06,USA,24.50
2,1004,Dana Lee,dana@example.com,Widget C,5.0,4.25,2026-01-07,South Africa,21.25
3,1006,Fay Osei,fay@example.com,Widget A,1.0,9.99,2026-01-08,South Africa,9.99
4,1007,Grace Kim,grace@example.com,Widget C,4.0,4.25,2026-01-08,USA,17.00
5,1009,Ivy Chen,ivy@example.com,Widget A,10.0,9.99,2026-01-09,South Africa,99.90
6,1010,Jon Park,jon@example.com,Widget C,3.0,4.25,2026-01-10,Kenya,12.75


### Step 3: Load
Write the clean data into a local SQLite warehouse.

In [3]:
import sqlite3

def load_orders(clean: pd.DataFrame, rejected: pd.DataFrame, db_path: str):
    con = sqlite3.connect(db_path)
    clean.to_sql("orders", con, if_exists="replace", index=False)
    rejected.to_sql("orders_rejected", con, if_exists="replace", index=False)
    con.commit()
    con.close()
    print(f"[load] wrote {len(clean)} clean rows, {len(rejected)} rejected rows to {db_path}")

load_orders(clean, rejected, "week01_warehouse.db")

[load] wrote 7 clean rows, 3 rejected rows to week01_warehouse.db


### Auto-grading checks
These are the kind of assertions a CI auto-grader would run against a forked submission.

In [4]:
assert len(raw) == 10, f"expected 10 raw rows, got {len(raw)}"
assert len(clean) == 7, f"expected 7 valid rows, got {len(clean)}"
assert len(rejected) == 3, f"expected 3 rejected rows, got {len(rejected)}"
assert "total_price" in clean.columns, "clean data must have a derived total_price column"
assert (clean["total_price"] == (clean["quantity"] * clean["unit_price"]).round(2)).all(), "total_price must equal quantity * unit_price"
assert set(rejected["order_id"]) == {"1003", "1005", "1008"}, "unexpected set of rejected order_ids"

con = sqlite3.connect("week01_warehouse.db")
loaded = pd.read_sql("SELECT * FROM orders", con)
con.close()
assert len(loaded) == len(clean), "warehouse table row count must match the clean DataFrame"

print("ALL CHECKS PASSED")

ALL CHECKS PASSED
